In [1]:
# Run this notebook under "SLUG/"
%cd ..

/media/yt/yt's volume/github_pub_space/SLUG


In [3]:
%load_ext autoreload
%autoreload 2

import os
import logging
import requests
import torch
from copy import deepcopy
from pathlib import Path

from PIL import Image
import numpy as np
from matplotlib import pyplot as plt
from tqdm import tqdm
from torch import nn
from transformers import CLIPProcessor, CLIPModel
from transformers import AutoProcessor, LlavaForConditionalGeneration
from accelerate import Accelerator

from src.vlm_util import get_dataset, eval_vlm

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
def create_wds(input_shards, bs=16):
    pipeline = [wds.SimpleShardList(input_shards)]
    pipeline.extend([
                wds.split_by_worker,
                wds.tarfile_to_samples(handler=log_and_continue),
                wds.select(filter_no_caption_or_no_image),
                wds.decode("pilrgb", handler=log_and_continue),
                wds.rename(image="jpg;png;jpeg;webp", text="txt"),
                # wds.map_dict(image=preprocess_img, text=lambda text: tokenizer(text)[0]),
                wds.to_tuple("image", "text"),
                wds.batched(bs, partial=True)
            ])

    dataset = wds.DataPipeline(*pipeline)

    dataloader = wds.WebLoader(
            dataset,
            batch_size=None,
            shuffle=False,
            num_workers=1,
            persistent_workers=True,
        )

    return dataloader

## Load CLIP

In [4]:
accelerator = Accelerator()
device = accelerator.device

clip_model_id = "openai/clip-vit-large-patch14-336"
model_clip = CLIPModel.from_pretrained(clip_model_id)
processor_clip = CLIPProcessor.from_pretrained(clip_model_id)

In [5]:
# celeb_name = "Tom_Cruise"
celeb_name = "Elon_Musk" # 0.0, 0.5
celeb_name = "Lady_Gaga" # 
celeb_name = "Mark_Zuckerberg" # 0.0, 0.8
celeb_name = "Taylor_Swift" # 0.0, 0.8

In [ ]:
model_clip.to(device)
model_clip.train()

model_repo, model_name = clip_model_id.split('/')
# update path to "openai/clip-vit-large-patch14-336" computed gradients
grad_root = Path(f"[...]/LLaVA/grads/{celeb_name}_{model_repo}_{model_name}") 
grad_file_name = grad_root/'forget_grads.pt'

if grad_file_name.exists:
    forget_grads = torch.load(grad_root/'forget_grads.pt', map_location='cpu')
    # update path to "openai/clip-vit-large-patch14-336" gradients
    retain_grads = torch.load('[...]/LLaVA/grads/openai_clip-vit-large-patch14-336/train_grads.pt', map_location='cpu')
else:
    for split in ['forget', 'train']:
        # if split == 'train' and celeb_name != 'Elon_Musk':
        #     continue
        gradients = dict([(n, torch.zeros_like(p, device=p.device)) for n, p in model_clip.named_parameters()])

        if split == 'forget':
            path = Path(f"data/laion/forget/names/{celeb_name}.tar")
        else:
            path = Path("data/laion/laion400m/00000.tar")
        dataloader = create_wds(str(path), bs=16)
        for i, (images, texts) in tqdm(enumerate(dataloader)):
            
            texts = [celeb_name.replace('_', ' ')] * len(texts)

            inputs = processor_clip(
                text=texts, images=images, return_tensors="pt", padding=True,
                truncation=True,      # Enable truncation
                max_length=77         # Set the maximum length to 77 tokens
            ).to(device)


            outputs = model_clip(**inputs, return_loss=True)
            image_features = outputs.image_embeds
            text_features = outputs.text_embeds
            if split == 'forget':
                total_loss = nn.CosineEmbeddingLoss()(image_features, text_features, torch.ones(len(images)).to(device))
            else:
                total_loss = outputs.loss

            total_loss.backward()
            
            # accululate gradients
            for name, param in model_clip.named_parameters():
                if param.grad is not None:
                    gradients[name] += param.grad
        
        
        # average the gradients
        for name, param in model_clip.named_parameters():
            if param.grad is not None:
                gradients[name] /= (i+1) # len(dataloader)

        # update path to "openai/clip-vit-large-patch14-336" computed gradients
        mask_save_root = Path(f"[...]/LLaVA/grads/{celeb_name}_{model_repo}_{model_name}")

        mask_save_root.mkdir(parents=True, exist_ok=True)
        torch.save(gradients, os.path.join(mask_save_root, f"{split}_grads.pt"))
        logging.info(f"Saved {split} gradients to {os.path.join(mask_save_root, f'{split}_grads.pt')}")

/tmp/ipykernel_559232/1946037443.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  forget_grads = torch.load(grad_root/'forget_grads.pt', map_location='cpu')
/tmp/ipykerne

In [ ]:
# identify important layers
from torch import nn
def identify_pareto(scores):
        # Initialize a list to store the index of Pareto points
        pareto_index = []
        # Loop through all points
        for i, (x, y) in enumerate(scores):
            dominated = False
            for j, (x2, y2) in enumerate(scores):
                # Check if point (x2, y2) dominates (x, y)
                if x2 < x and y2 > y:
                    dominated = True
                    break
            if not dominated:
                pareto_index.append(i)
        return pareto_index

def get_important_layers(celeb_name, pair, model, base_root='clip/grads/name'):
    model_name, ckpt = pair.split('/')
    mask_root = Path(f'{base_root}/{celeb_name}_{model_name}_{ckpt}')
    retain_mask_root = Path(f'{base_root}/{model_name}_{ckpt}')
    forget_importances = torch.load(mask_root/'forget_grads.pt', map_location='cpu')
    retain_importances = torch.load(retain_mask_root/'train_grads.pt', map_location='cpu')
    
    # get model parameters
    model_params = {}
    for idx, (k, p) in enumerate(model.named_parameters()):
        model_params[k] = p.data
    
    # get forget importance ratio
    forget_ratio_dict = {}
    for layer_name in model_params:
        params_norm = torch.norm(model_params[layer_name]).item()
        grad_norm = torch.norm(forget_importances[layer_name]).item()
        if grad_norm > 0:
            forget_ratio_dict[layer_name] = grad_norm / params_norm
        # forget_ratio_dict[layer_name] = (forget_importances[layer_name] / model_params[layer_name]).abs().mean()
    # sort
    ranked_forget_ratio = {k: v for k, v in sorted(forget_ratio_dict.items(), key=lambda item: item[1], reverse=True)}

    cos = nn.CosineSimilarity(dim=0, eps=1e-6)
    cosine_dict = {}
    for layer_name in model_params:
        if len(retain_importances[layer_name].shape) > 0:
            # cosine_dict[layer_name] = cos(retain_importances[layer_name].flatten(), forget_importances[layer_name].flatten())
            cosine_dict[layer_name] = abs(cos(retain_importances[layer_name].flatten(), forget_importances[layer_name].flatten())).item()
    ranked_cos_name_list = []
    ranked_cos = {k: v for k, v in sorted(cosine_dict.items(), key=lambda item: item[1], reverse=True)}

    important_layers = {}
    save_root = Path(f'clip/figs/output/{celeb_name}/')
    save_root.mkdir(parents=True, exist_ok=True)
    # import pdb; pdb.set_trace()

    # for part in ['vision', 'language']:
    # for part in ['language']: # SD uses CLIP text encoder only
    for part in ['vision']: # LLaVA-VLM uses CLIP text encoder only
        # make plot
        name_list = []
        x_cos_list = []
        y_ratio_list = []
        for key in ranked_cos:
            if "bias" in key: continue
            if 'logit_scale' in key: continue
            if 'position' in key: continue
            if 'embedding' in key: continue
            if 'norm' in key: continue
            # if '.ln_' in key: continue
            if part == "vision" and "vision" not in key: continue
            if part != "vision" and "vision" in key: continue
            
            name_list.append(key)
            x_cos_list.append(ranked_cos[key])
            y_ratio_list.append(ranked_forget_ratio[key])
        
        
        # Use the function to find Pareto front
        pareto_indices = identify_pareto(list(zip(x_cos_list, y_ratio_list)))

        font_size = 12
        line_width = 3
        fig = plt.figure()
        # ax = fig.add_subplot(111)

        for idx, (name, x, y) in enumerate(zip(name_list, x_cos_list, y_ratio_list)):
            # if name in ranked_forget_ratio_name_list[:5] or name in ranked_cos_name_list[-5:]:
            if idx in pareto_indices:
                if part not in important_layers:
                    important_layers[part] = [name]
                else:
                    important_layers[part].append(name)
                # plt.scatter(x, y, label=name)
                if part == 'vision':
                    plt.scatter(x, y, label=name.replace('visual.transformer.resblocks.', '').replace('.weight', '').replace('_weight', ''))
                else:
                    plt.scatter(x, y, label=name.replace('transformer.resblocks.', '').replace('.weight', '').replace('_weight', ''))
            else:
                plt.scatter(x, y, marker='x', c='k')
        plt.xscale('log')
        plt.yscale('log')

        # # Set tick parameters with larger font size and bold weight
        # ax.tick_params(axis='both', which='major', labelsize=font_size, width=line_width)
        # for label in ax.get_xticklabels() + ax.get_yticklabels():
        #     label.set_fontsize(font_size)
        #         # label.set_fontweight('bold')


        # plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), prop={'size': 10})
        plt.legend(loc='lower left', bbox_to_anchor=(0, 0), prop={'size': 10}, fancybox=True, framealpha=0.5)
        # plt.title(f"[{celeb_name}] Layers on Pareto Front (for Vision)")
        # plt.xlabel("cosine similarity between forget and retain gradients")
        plt.xlabel("Gradient Alignment", fontsize=font_size, weight='bold')
        # plt.ylabel("ratio of forget gradients and model weights")
        plt.ylabel("Importance of Layers", fontsize=font_size, weight='bold')

        plt.tight_layout()
        plt.savefig(save_root/f'pareto-{part}-{celeb_name}.png')
        plt.close()

    return important_layers

In [ ]:
important_layers = get_important_layers(celeb_name, clip_model_id, model_clip, base_root='[...]/LLaVA/grads/')
important_layers

/tmp/ipykernel_559232/234449780.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  forget_importances = torch.load(mask_root/'forget_grads.pt', map_location='cpu')
/tmp/ip

{'vision': ['vision_model.encoder.layers.22.self_attn.v_proj.weight',
  'vision_model.encoder.layers.21.self_attn.v_proj.weight',
  'vision_model.encoder.layers.12.mlp.fc2.weight',
  'vision_model.encoder.layers.2.self_attn.q_proj.weight']}

In [ ]:
layer_num_max = 0
layer_name_max = None
# select the deepest layer
for layer_name in important_layers['vision']:
    # skip norm and other layers
    if not 'vision_model.encoder.layers' in layer_name: continue
    layer_num = int(layer_name.split('.')[3])
    if layer_num > layer_num_max:
        layer_num_max = layer_num
        layer_name_max = layer_name
if layer_name_max is None:
    # pick an empirically good layer
    layer_name_max = "text_model.encoder.layers.11.self_attn.out_proj.weight"
layer_name = layer_name_max

# Make sure update the whole layer
layer_component = layer_name.split('.')[5]
if layer_component in ['k_proj', 'v_proj', 'q_proj']:
    layer_names = [layer_name.replace(layer_component, attn) for attn in ('k_proj', 'v_proj', 'q_proj')]
elif layer_component in ['fc1', 'fc2']:
    layer_names = [layer_name.replace(layer_component, attn) for attn in ('fc1', 'fc2')]
else:
    layer_names = [layer_name]   

vector = forget_grads[layer_name].to(device)

# get weight norm and ratio
params_norm = torch.norm(model_clip.get_parameter(layer_name)).item()
grad_norm = torch.norm(vector).item()
ratio = params_norm/grad_norm
print(f"Layer name: {layer_name}")
print(f"params_norm: {params_norm}")
print(f"grad_norm: {grad_norm}")
print(f"ratio: {ratio}")

Layer name: vision_model.encoder.layers.22.self_attn.v_proj.weight
params_norm: 15.55045223236084
grad_norm: 19.330875396728516
ratio: 0.8044360078484878


In [10]:
model_clip.get_parameter(layer_name).data

tensor([[-0.0094,  0.0040, -0.0081,  ...,  0.0087,  0.0034, -0.0086],
        [-0.0218, -0.0027,  0.0096,  ...,  0.0072, -0.0114, -0.0138],
        [-0.0069,  0.0196,  0.0128,  ...,  0.0089, -0.0430,  0.0282],
        ...,
        [ 0.0080, -0.0076,  0.0247,  ..., -0.0006,  0.0264, -0.0060],
        [-0.0081,  0.0084,  0.0222,  ...,  0.0080, -0.0070,  0.0124],
        [-0.0113,  0.0073,  0.0137,  ..., -0.0100, -0.0085,  0.0073]],
       device='cuda:0')

In [11]:
model_clip_cpu = model_clip.to('cpu')
del model_clip
torch.cuda.empty_cache()
# model_clip_pretrained = deepcopy(model_clip)

## Load VLM

In [ ]:
accelerator = Accelerator()
device = accelerator.device
# load vlm and run
model_pretrained = LlavaForConditionalGeneration.from_pretrained("llava-hf/llava-1.5-7b-hf",torch_dtype=torch.float16)
llava_processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")


# load vlm dataset
target_id_dataset = get_dataset("ytan-ucr/mu_llava_tom_cruise", subset_size=10)
celeb_datatset = get_dataset("ytan-ucr/mu_llava_celeb", subset_size=10)

Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  5.04it/s]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [13]:
# model_pretrained.vision_tower.get_parameter(layer_name).data

llava_model = model_pretrained.to(device)

tgt_fgt_acc_original = eval_vlm(llava_model,llava_processor, target_id_dataset)
celeb_fgt_acc_original = eval_vlm(llava_model,llava_processor, celeb_datatset)


model_pretrained = model_pretrained.to('cpu')
del llava_model
torch.cuda.empty_cache()

tgt_fgt_acc_original, celeb_fgt_acc_original

100%|██████████| 10/10 [00:04<00:00,  2.20it/s]


(1.0, 0.9)

In [14]:
layer_data_original = deepcopy(model_pretrained.vision_tower.get_parameter(layer_name).data).to(device)

In [ ]:
vlm_model_name = 'llava'
part = 'vision' # llava use only vision clip

save_root = Path(f'../results/slug_vlm/')
save_root.mkdir(parents=True, exist_ok=True)
# Binary search algorithm
# Constants
INITIAL_RATIO_DIVISOR = 10
MAX_ITERATIONS = 10
TOLERANCE = 0.01

# Initialize variables
cnt = 0  # Search count

info = f"iter: {cnt}, ratio: 0, fgt_acc_tgt: {tgt_fgt_acc_original}, fgt_acc_celeb: {celeb_fgt_acc_original}\n"
logging.info(info)
print(info)
# save to txt
with open(save_root/f'log_{vlm_model_name}-{part}-{layer_name}.txt', 'a') as f:
    f.write(f"{info}\n")

# Main loop for adjusting the ratio
while cnt < MAX_ITERATIONS:
    
    if cnt == 0:
        # Start with 1/10 of the norm ratio
        ratio = - (ratio / INITIAL_RATIO_DIVISOR)
        ratio_low = 0
        ratio_high = float('inf')
        print(f"Start with ratio: {ratio}")
    else:
        if tgt_id_fgt_acc == 0:
            # Reduce the gradient
            ratio_high = ratio
            ratio = (ratio_low + ratio_high) / 2
            print(f"[Reduce ratio] Iteration: {cnt}, Ratio: {ratio}, Ratio_low: {ratio_low}, Ratio_high: {ratio_high}")

        elif tgt_id_fgt_acc > 0:
            # Magnify the gradient
            ratio_low = ratio
            if ratio_high != float('inf'):
                ratio = (ratio_low + ratio_high) / 2
                print(f"[Increase ratio] Iteration: {cnt}, Ratio: {ratio}, Ratio_low: {ratio_low}, Ratio_high: {ratio_high}")
            else:
                ratio = ratio * 2
                print(f"[Increase ratio] Iteration: {cnt}, Ratio: {ratio}, Ratio_low: {ratio_low}, Ratio_high: None")


    print(f"iter: {cnt}, ratio: {ratio}")
    model = deepcopy(model_pretrained).to(device)

    # llava_model.vision_tower.vision_model === model_clip.vision_model
    # model.vision_tower.get_parameter(layer_name).data = model_pretrained.vision_tower.get_parameter(layer_name).data + ratio*vector
    model.vision_tower.get_parameter(layer_name).data = layer_data_original + ratio*vector.half()

    cnt += 1  # Increment the search count
    epoch = cnt

    tgt_id_fgt_acc = eval_vlm(model,llava_processor, target_id_dataset)
    all_celeb_fgt_acc = eval_vlm(model,llava_processor, celeb_datatset)
    # forget_acc1, forget_acc5, celeb100_top1, celeb100_top5, test_top1, test_top5, MIA_mean, MIA_std = evaluate_model(model, data, epoch, args, tokenizer, preprocess=preprocess_val, celeb_name=args.celeb_name)
    model.to('cpu')
    del model
    torch.cuda.empty_cache()

    info = f"iter: {cnt}, ratio: {ratio}, fgt_acc_tgt: {tgt_id_fgt_acc}, fgt_acc_celeb: {all_celeb_fgt_acc}\n"
    logging.info(info)
    # info = f"iter: {cnt}, ratio: {ratio}, fgt_acc@1: {forget_acc1}, fgt_acc@5: {forget_acc5}, test_acc@1: {test_top1}, test_acc@5: {test_top5}"
    print(info)
    
    # save to txt
    with open(save_root/f'log_{vlm_model_name}-{part}-{layer_name}.txt', 'a') as f:
        f.write(f"{info}\n")

iter: 0, ratio: 0, fgt_acc_tgt: 1.0, fgt_acc_celeb: 0.9

Start with ratio: -0.08044360078484877
iter: 0, ratio: -0.08044360078484877


100%|██████████| 10/10 [00:04<00:00,  2.12it/s]


iter: 1, ratio: -0.08044360078484877, fgt_acc_tgt: 0.6, fgt_acc_celeb: 1.0

[Increase ratio] Iteration: 1, Ratio: -0.16088720156969755, Ratio_low: -0.08044360078484877, Ratio_high: None
iter: 1, ratio: -0.16088720156969755


100%|██████████| 10/10 [00:04<00:00,  2.06it/s]


iter: 2, ratio: -0.16088720156969755, fgt_acc_tgt: 0.0, fgt_acc_celeb: 0.7

[Reduce ratio] Iteration: 2, Ratio: -0.12066540117727316, Ratio_low: -0.08044360078484877, Ratio_high: -0.16088720156969755
iter: 2, ratio: -0.12066540117727316


...

iter: 8, ratio: -0.10683915729237728, fgt_acc_tgt: 0.1, fgt_acc_celeb: 0.8

[Increase ratio] Iteration: 8, Ratio: -0.10746762292350892, Ratio_low: -0.10683915729237728, Ratio_high: -0.10809608855464055
iter: 8, ratio: -0.10746762292350892


100%|██████████| 10/10 [00:04<00:00,  2.03it/s]


iter: 9, ratio: -0.10746762292350892, fgt_acc_tgt: 0.1, fgt_acc_celeb: 0.8

[Increase ratio] Iteration: 9, Ratio: -0.10778185573907473, Ratio_low: -0.10746762292350892, Ratio_high: -0.10809608855464055
iter: 9, ratio: -0.10778185573907473


100%|██████████| 10/10 [00:04<00:00,  2.02it/s]


iter: 10, ratio: -0.10778185573907473, fgt_acc_tgt: 0.0, fgt_acc_celeb: 0.8

